# Olist — Feature Engineering

Processa os dados do e-commerce Olist e gera um dataset pronto para treinar um modelo de Machine Learning que prevê quantos dias um pedido leva para ser entregue.

**Fonte:** [Olist Dataset — Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Target:** `dias_entrega` — dias entre a compra e a entrega ao cliente

## 1. Importações e Configurações

In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

PASTA = "C:/Users/eymar/Downloads/olist/archive/"

print("Bibliotecas importadas.")

Bibliotecas importadas.


## 2. Carregamento dos Dados

In [2]:
orders    = pd.read_csv(PASTA + 'olist_orders_dataset.csv')
items     = pd.read_csv(PASTA + 'olist_order_items_dataset.csv')
products  = pd.read_csv(PASTA + 'olist_products_dataset.csv')
customers = pd.read_csv(PASTA + 'olist_customers_dataset.csv')
sellers   = pd.read_csv(PASTA + 'olist_sellers_dataset.csv')
payments  = pd.read_csv(PASTA + 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(PASTA + 'olist_order_reviews_dataset.csv')
geo       = pd.read_csv(PASTA + 'olist_geolocation_dataset.csv')
traducao  = pd.read_csv(PASTA + 'product_category_name_translation.csv')

tabelas = {
    'orders': orders, 'items': items, 'products': products,
    'customers': customers, 'sellers': sellers, 'payments': payments,
    'reviews': reviews, 'geo': geo, 'traducao': traducao
}
for nome, df in tabelas.items():
    print(f"{nome:12s} -> {df.shape[0]:>8,} linhas | {df.shape[1]} colunas")

orders       ->   99,441 linhas | 8 colunas
items        ->  112,650 linhas | 7 colunas
products     ->   32,951 linhas | 9 colunas
customers    ->   99,441 linhas | 5 colunas
sellers      ->    3,095 linhas | 4 colunas
payments     ->  103,886 linhas | 5 colunas
reviews      ->   99,224 linhas | 7 colunas
geo          -> 1,000,163 linhas | 5 colunas
traducao     ->       71 linhas | 2 colunas


## 3. Pré-processamento dos Pedidos (Target)

In [3]:
# Converte colunas de data para datetime
for col in ['order_purchase_timestamp', 'order_approved_at',
            'order_delivered_carrier_date', 'order_delivered_customer_date',
            'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col])

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

# Mantém só pedidos entregues e sem datas faltando
orders = orders[orders['order_status'] == 'delivered'].copy()
orders = orders.dropna(subset=['order_delivered_customer_date',
                                'order_approved_at',
                                'order_delivered_carrier_date'])

# Target: dias entre a compra e a entrega
orders['dias_entrega'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

# Remove outliers
antes = len(orders)
orders = orders[(orders['dias_entrega'] > 0) & (orders['dias_entrega'] <= 120)]
depois = len(orders)

print(f"Pedidos apos filtro: {depois:,}  (removidos {antes - depois:,} outliers)")
print(f"Media:   {orders['dias_entrega'].mean():.1f} dias")
print(f"Mediana: {orders['dias_entrega'].median():.1f} dias")
orders[['dias_entrega']].describe().round(2)

Pedidos apos filtro: 96,412  (removidos 43 outliers)
Media:   12.5 dias
Mediana: 10.2 dias


,dias_entrega
count,96412.00
mean,12.49
std,8.99
min,0.53
25%,6.76
50%,10.21
75%,15.71
max,118.97


## 4. Features do Pedido (Timestamp)

In [4]:
# Extrai informacoes do momento da compra
orders['hora_compra']       = orders['order_purchase_timestamp'].dt.hour
orders['dia_semana_compra'] = orders['order_purchase_timestamp'].dt.dayofweek
orders['mes_compra']        = orders['order_purchase_timestamp'].dt.month
orders['eh_fim_de_semana']  = (orders['dia_semana_compra'] >= 5).astype(int)

# Tempo entre etapas do pedido (em horas)
orders['horas_ate_aprovacao'] = (
    orders['order_approved_at'] - orders['order_purchase_timestamp']
).dt.total_seconds() / 3600

orders['horas_ate_transportadora'] = (
    orders['order_delivered_carrier_date'] - orders['order_approved_at']
).dt.total_seconds() / 3600

orders[['hora_compra','dia_semana_compra','mes_compra','eh_fim_de_semana',
        'horas_ate_aprovacao','horas_ate_transportadora']].describe().round(2)

,hora_compra,dia_semana_compra,mes_compra,eh_fim_de_semana,horas_ate_aprovacao,horas_ate_transportadora
count,96412.00,96412.00,96412.00,96412.00,96412.00,96412.00
mean,14.77,2.76,6.03,0.23,10.28,67.12
std,5.33,1.97,3.23,0.42,20.54,84.30
min,0.00,0.00,1.00,0.00,0.00,-4109.26
25%,11.00,1.00,3.00,0.00,0.22,20.98
50%,15.00,3.00,6.00,0.00,0.34,43.57
75%,19.00,4.00,8.00,0.00,14.52,85.78
max,23.00,6.00,12.00,1.00,741.44,2569.28


## 5. Features dos Itens do Pedido

In [5]:
# Agrega itens por pedido
feats_itens = items.groupby('order_id').agg(
    qtd_itens          = ('order_item_id', 'count'),
    qtd_vendedores     = ('seller_id',     'nunique'),
    valor_total_pedido = ('price',         'sum'),
    valor_total_frete  = ('freight_value', 'sum'),
).reset_index()

feats_itens['multiplos_vendedores'] = (feats_itens['qtd_vendedores'] > 1).astype(int)
feats_itens['proporcao_frete']      = (
    feats_itens['valor_total_frete'] / feats_itens['valor_total_pedido']
).round(4)

print(f"{feats_itens.shape[0]:,} pedidos")
feats_itens.drop(columns='order_id').describe().round(2)

98,666 pedidos


,qtd_itens,qtd_vendedores,valor_total_pedido,valor_total_frete,multiplos_vendedores,proporcao_frete
count,98666.00,98666.00,98666.00,98666.00,98666.00,98666.00
mean,1.14,1.01,137.75,22.82,0.01,0.31
std,0.54,0.12,210.65,21.65,0.11,0.31
min,1.00,1.00,0.85,0.00,0.00,0.00
25%,1.00,1.00,45.90,13.85,0.00,0.13
50%,1.00,1.00,86.90,17.17,0.00,0.22
75%,1.00,1.00,149.90,24.04,0.00,0.38
max,21.00,5.00,13440.00,1794.96,1.00,21.45


## 6. Features dos Produtos

In [6]:
# Adiciona traducao de categoria e calcula volume
products = products.merge(traducao, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('desconhecido')
products['volume_cm3']    = products['product_length_cm'] * products['product_height_cm'] * products['product_width_cm']
products['tem_descricao'] = (products['product_description_lenght'] > 0).astype(int)

items_prod = items.merge(products, on='product_id', how='left')

feats_prod = items_prod.groupby('order_id').agg(
    peso_total_gramas = ('product_weight_g',              'sum'),
    volume_total_cm3  = ('volume_cm3',                    'sum'),
    media_qtd_fotos   = ('product_photos_qty',            'mean'),
    categoria_produto = ('product_category_name_english', lambda x: x.mode()[0] if len(x) > 0 else 'desconhecido'),
    tem_descricao     = ('tem_descricao',                 'max'),
).reset_index()

feats_prod['peso_por_item']     = (feats_prod['peso_total_gramas'] / feats_prod['peso_total_gramas'].replace(0, np.nan)).fillna(0)
feats_prod['peso_total_gramas'] = feats_prod['peso_total_gramas'].fillna(feats_prod['peso_total_gramas'].median())
feats_prod['volume_total_cm3']  = feats_prod['volume_total_cm3'].fillna(feats_prod['volume_total_cm3'].median())
feats_prod['media_qtd_fotos']   = feats_prod['media_qtd_fotos'].fillna(feats_prod['media_qtd_fotos'].median())

print(f"{feats_prod.shape[0]:,} pedidos")
print(feats_prod['categoria_produto'].value_counts().head())
feats_prod.drop(columns=['order_id','categoria_produto']).describe().round(2)

98,666 pedidos
categoria_produto
bed_bath_table           9384
health_beauty            8808
sports_leisure           7664
computers_accessories    6679
furniture_decor          6348
Name: count, dtype: int64


,peso_total_gramas,volume_total_cm3,media_qtd_fotos,tem_descricao,peso_por_item
count,98666.00,98666.00,98666.00,98666.00,98666.00
mean,2390.03,17401.43,2.25,0.99,1.00
std,4773.24,30402.17,1.73,0.12,0.01
min,0.00,0.00,1.00,0.00,0.00
25%,300.00,2964.00,1.00,1.00,1.00
50%,750.00,7260.00,2.00,1.00,1.00
75%,2066.75,19872.00,3.00,1.00,1.00
max,184400.00,1476000.00,20.00,1.00,1.00


## 7. Features Geograficas

In [7]:
def haversine(lat1, lon1, lat2, lon2):
    """Distancia em km entre dois pontos geograficos."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# Media de coordenadas por CEP (geolocation tem ~1M linhas)
geo_media = geo.groupby('geolocation_zip_code_prefix')[
    ['geolocation_lat', 'geolocation_lng']].mean().reset_index()

# Coordenadas do cliente
cust_geo = customers.merge(
    geo_media, left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix', how='left'
).rename(columns={'geolocation_lat': 'lat_cliente', 'geolocation_lng': 'lng_cliente'})

# Vendedor principal por pedido
seller_por_pedido = items[['order_id', 'seller_id']].drop_duplicates('order_id')

# Coordenadas do vendedor
seller_geo = seller_por_pedido.merge(sellers, on='seller_id', how='left').merge(
    geo_media, left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix', how='left'
).rename(columns={'geolocation_lat': 'lat_vendedor', 'geolocation_lng': 'lng_vendedor'})

# Calcula distancia e flags geograficas
geo_feats = orders[['order_id', 'customer_id']].merge(
    cust_geo[['customer_id', 'customer_state', 'lat_cliente', 'lng_cliente']],
    on='customer_id', how='left'
).merge(
    seller_geo[['order_id', 'seller_id', 'seller_state', 'lat_vendedor', 'lng_vendedor']],
    on='order_id', how='left'
)

mask = geo_feats['lat_cliente'].notna() & geo_feats['lat_vendedor'].notna()
geo_feats['distancia_km'] = np.nan
geo_feats.loc[mask, 'distancia_km'] = geo_feats[mask].apply(
    lambda r: haversine(r.lat_cliente, r.lng_cliente, r.lat_vendedor, r.lng_vendedor), axis=1
)

geo_feats['mesmo_estado'] = (geo_feats['customer_state'] == geo_feats['seller_state']).astype(int)

regioes = {
    'AC':'Norte','AM':'Norte','AP':'Norte','PA':'Norte','RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MS':'Centro-Oeste','MT':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul'
}
geo_feats['regiao_cliente'] = geo_feats['customer_state'].map(regioes).fillna('desconhecido')
geo_feats['distancia_km']   = geo_feats['distancia_km'].fillna(geo_feats['distancia_km'].median())

print(f"Distancia media: {geo_feats['distancia_km'].mean():.0f} km")
print(f"Mesmo estado:    {geo_feats['mesmo_estado'].mean()*100:.1f}% dos pedidos")
geo_feats[['distancia_km','mesmo_estado']].describe().round(2)

Distancia media: 600 km
Mesmo estado:    36.0% dos pedidos


,distancia_km,mesmo_estado
count,96412.00,96412.00
mean,599.64,0.36
std,592.08,0.48
min,0.00,0.00
25%,188.91,0.00
50%,433.79,0.00
75%,795.55,1.00
max,8677.91,1.00


## 8. Features de Pagamento

In [8]:
# Agrega pagamentos por pedido
feats_pag = payments.groupby('order_id').agg(
    tipo_pagamento      = ('payment_type',         lambda x: x.mode()[0]),
    qtd_parcelas        = ('payment_installments',  'max'),
    valor_pago_total    = ('payment_value',          'sum'),
    qtd_tipos_pagamento = ('payment_type',           'nunique'),
).reset_index()

print(feats_pag['tipo_pagamento'].value_counts())
feats_pag[['qtd_parcelas','valor_pago_total','qtd_tipos_pagamento']].describe().round(2)

tipo_pagamento
credit_card    76132
boleto         19784
voucher         1994
debit_card      1527
not_defined        3
Name: count, dtype: int64


,qtd_parcelas,valor_pago_total,qtd_tipos_pagamento
count,99440.00,99440.00,99440.00
mean,2.93,160.99,1.02
std,2.72,221.95,0.15
min,0.00,0.00,1.00
25%,1.00,62.01,1.00
50%,2.00,105.29,1.00
75%,4.00,176.97,1.00
max,24.00,13664.08,2.00


## 9. Features Históricas do Vendedor

In [9]:
# Media historica de entrega por vendedor
hist_vendedor = orders.merge(seller_por_pedido, on='order_id', how='left').groupby('seller_id').agg(
    media_dias_entrega_vendedor = ('dias_entrega', 'mean'),
    total_pedidos_vendedor      = ('order_id',     'count'),
).reset_index()
hist_vendedor['media_dias_entrega_vendedor'] = hist_vendedor['media_dias_entrega_vendedor'].round(2)

# Media de avaliacao por vendedor
feats_seller_review = reviews.merge(
    orders[['order_id']], on='order_id', how='inner'
).merge(seller_por_pedido, on='order_id', how='left').groupby('seller_id').agg(
    media_avaliacao_vendedor = ('review_score', 'mean'),
).reset_index()

feats_hist = hist_vendedor.merge(feats_seller_review, on='seller_id', how='left')
feats_hist['media_avaliacao_vendedor'] = feats_hist['media_avaliacao_vendedor'].fillna(
    feats_hist['media_avaliacao_vendedor'].median()
)

print(f"{feats_hist.shape[0]:,} vendedores")
feats_hist[['media_dias_entrega_vendedor','total_pedidos_vendedor','media_avaliacao_vendedor']].describe().round(2)

2,959 vendedores


,media_dias_entrega_vendedor,total_pedidos_vendedor,media_avaliacao_vendedor
count,2959.00,2959.00,2959.00
mean,12.10,32.58,4.20
std,6.20,104.36,0.77
min,1.21,1.00,1.00
25%,8.41,2.00,4.00
50%,11.16,7.00,4.32
75%,14.18,22.00,4.71
max,86.00,1809.00,5.00


## 10. Montagem do Dataset Final

In [10]:
# Junta todos os grupos de features
df = orders[['order_id', 'customer_id', 'dias_entrega', 'hora_compra',
             'dia_semana_compra', 'mes_compra', 'eh_fim_de_semana',
             'horas_ate_aprovacao', 'horas_ate_transportadora']].copy()

df = df.merge(feats_itens, on='order_id', how='left')
df = df.merge(feats_prod,  on='order_id', how='left')
df = df.merge(geo_feats[['order_id', 'seller_id', 'customer_state', 'seller_state',
                          'mesmo_estado', 'distancia_km', 'regiao_cliente']],
              on='order_id', how='left')
df = df.merge(feats_pag,  on='order_id', how='left')
df = df.merge(feats_hist, on='seller_id', how='left')

# Preenche nulos numericos com a mediana
for col in df.select_dtypes(include='number').columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(f"Linhas:  {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")
print(f"Nulos:   {df.isnull().sum().sum()}")
df.head(3)

Linhas:  96,412
Colunas: 34
Nulos:   1


,order_id,customer_id,dias_entrega,hora_compra,dia_semana_compra,mes_compra,eh_fim_de_semana,horas_ate_aprovacao,horas_ate_transportadora,qtd_itens,...,mesmo_estado,distancia_km,regiao_cliente,tipo_pagamento,qtd_parcelas,valor_pago_total,qtd_tipos_pagamento,media_dias_entrega_vendedor,total_pedidos_vendedor,media_avaliacao_vendedor
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,8.436574,10,0,10,0,0.178333,56.795833,1,...,1,18.576110,Sudeste,voucher,1.0,38.71,2.0,7.38,52,4.480769
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,13.782037,20,1,7,0,30.713889,11.109167,1,...,0,851.495069,Nordeste,boleto,1.0,141.46,1.0,6.76,104,4.605769
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,9.394213,8,2,8,0,0.276111,4.910278,1,...,0,514.410666,Centro-Oeste,credit_card,3.0,179.12,1.0,15.09,1117,4.164112


## 11. Resumo das Colunas por Grupo

In [11]:
grupos = {
    "Pedido":     ['dias_entrega','hora_compra','dia_semana_compra','mes_compra','eh_fim_de_semana','horas_ate_aprovacao','horas_ate_transportadora'],
    "Itens":      ['qtd_itens','qtd_vendedores','valor_total_pedido','valor_total_frete','multiplos_vendedores','proporcao_frete'],
    "Produto":    ['peso_total_gramas','volume_total_cm3','media_qtd_fotos','categoria_produto','tem_descricao','peso_por_item'],
    "Geografia":  ['customer_state','seller_state','mesmo_estado','distancia_km','regiao_cliente'],
    "Pagamento":  ['tipo_pagamento','qtd_parcelas','valor_pago_total','qtd_tipos_pagamento'],
    "Vendedor":   ['media_dias_entrega_vendedor','total_pedidos_vendedor','media_avaliacao_vendedor'],
}

for grupo, cols in grupos.items():
    print(f"\n[{grupo}]")
    for col in cols:
        print(f"  {col}")


[Pedido]
  dias_entrega
  hora_compra
  dia_semana_compra
  mes_compra
  eh_fim_de_semana
  horas_ate_aprovacao
  horas_ate_transportadora

[Itens]
  qtd_itens
  qtd_vendedores
  valor_total_pedido
  valor_total_frete
  multiplos_vendedores
  proporcao_frete

[Produto]
  peso_total_gramas
  volume_total_cm3
  media_qtd_fotos
  categoria_produto
  tem_descricao
  peso_por_item

[Geografia]
  customer_state
  seller_state
  mesmo_estado
  distancia_km
  regiao_cliente

[Pagamento]
  tipo_pagamento
  qtd_parcelas
  valor_pago_total
  qtd_tipos_pagamento

[Vendedor]
  media_dias_entrega_vendedor
  total_pedidos_vendedor
  media_avaliacao_vendedor


## 12. Salvar Dataset Final

In [12]:
OUTPUT = PASTA + 'dataset_features.csv'
df.to_csv(OUTPUT, index=False)

print(f"Salvo em: {OUTPUT}")
print(f"Linhas:   {df.shape[0]:,}")
print(f"Colunas:  {df.shape[1]}")

Salvo em: C:/Users/eymar/Downloads/olist/archive/dataset_features.csv
Linhas:   96,412
Colunas:  34
